# agentic_alpha_tail_risk_external_search_v1

Diagnostic R-G-E-V-D card for an external tail-risk branch. It applies the Agentic Alpha workflow to VIX term-structure, VVIX, and SKEW evidence, builds a qlib provider from local external data, validates it through `D.features`, and closes the branch against the production `qlib_paper_sim_v1` return and MaxDD gate.


In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from qlib.data import D
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.qlib_native import QlibProviderConfig, build_qlib_store_from_frame, load_qlib_feature_frame
from src.factor import build_tail_risk_external_frame, evaluate_tail_risk_candidate, summarize_tail_risk_branch
from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_tail_risk_external_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl_tail_risk_external_v1').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production' / 'qlib_paper_sim_v1'
RAW_EXTERNAL = ROOT / 'data' / 'raw' / 'external'
PANEL_PATH = ROOT / 'data' / 'processed' / 'panels' / 'soxl_soxx_vix_full.csv'
START = '2019-01-02'
SOURCE_END = '2026-04-17'
MAX_DRAWDOWN_FLOOR = -20.0
COST_PER_TURNOVER = 0.0002

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)
if PROVIDER_URI.exists():
    shutil.rmtree(PROVIDER_URI)

baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')

workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: diagnostic_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  external_tail_risk:
    start: {START}
    source_end: {SOURCE_END}
rgevd:
  source: https://github.com/JacobDu/agentic-alpha
  stages:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  baseline_return_pct: {baseline.return_pct}
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Use Research/Retrieve/Generate/Evaluate/Validate/Distill to close each generated branch with evidence.',
            'retrieval_status': 'workflow_reference',
        },
        {
            'source_type': 'paper',
            'title': 'Risk Premia and the VIX Term Structure',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2548050',
            'extracted_signal': 'VIX term-structure slope can summarize volatility risk-premium states and return opportunities.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'The SKEW Index: Extracting What Has Been Left',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3121027',
            'extracted_signal': 'SKEW proxies downside tail-risk expectations and can warn about future market downturn states.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Volatility of Volatility and Tail Risk Premiums',
            'url': 'https://www.federalreserve.gov/pubs/feds/2013/201354/index.html',
            'extracted_signal': 'VVIX represents volatility-of-volatility and tail-risk hedge demand, so it is tested as a crash-risk throttle.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Cboe VIX Term Structure',
            'url': 'https://www.cboe.com/tradable-products/vix/term-structure/',
            'extracted_signal': 'VIX9D, VIX, and VIX3M jointly describe short-vs-medium volatility sentiment and backwardation.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'arxiv',
            'title': 'Compounding effects in leveraged ETFs',
            'url': 'https://arxiv.org/html/2504.20116v1',
            'extracted_signal': 'LETF path returns depend on trend persistence versus volatility clustering, motivating risk throttles.',
            'retrieval_status': 'prior_retrieval_seed',
        },
        {
            'source_type': 'twitter',
            'title': 'SOXL drawdown and volatility discussion seed',
            'url': 'https://x.com/ces921/highlights',
            'extracted_signal': 'Treat SOXL as regime-sensitive; volatility and drawdown state must gate raw leverage.',
            'retrieval_status': 'search_seed',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)

panel = pd.read_csv(PANEL_PATH)
vix9d = pd.read_csv(RAW_EXTERNAL / 'VIX9D.csv')
vix3m = pd.read_csv(RAW_EXTERNAL / 'VIX3M.csv')
vvix = pd.read_csv(RAW_EXTERNAL / 'VVIX.csv')
skew = pd.read_csv(RAW_EXTERNAL / 'SKEW.csv')
feature_frame = build_tail_risk_external_frame(
    panel=panel,
    vix9d=vix9d,
    vix3m=vix3m,
    vvix=vvix,
    skew=skew,
    start=START,
    end=SOURCE_END,
    zscore_min_periods=42,
)

qlib_store_frame = feature_frame.copy()
qlib_store_frame['instrument'] = 'SOXL'
field_map = {
    'close': 'soxl_close',
    'volume': 'soxl_volume',
    'soxx_close': 'soxx_close',
    'soxx_volume': 'soxx_volume',
    'vix': 'vix',
    'vix9d': 'vix9d',
    'vix3m': 'vix3m',
    'vvix': 'vvix',
    'skew': 'skew',
    'trend_20': 'trend_20',
    'trend_60': 'trend_60',
    'hv20': 'hv20',
    'vix9d_vix3m': 'vix9d_vix3m',
    'vix_vix3m': 'vix_vix3m',
    'vvix_vix': 'vvix_vix',
    'vix9d_vix3m_z': 'vix9d_vix3m_z',
    'vix_vix3m_z': 'vix_vix3m_z',
    'vvix_vix_z': 'vvix_vix_z',
    'skew_z': 'skew_z',
    'vix_z': 'vix_z',
}
provider_manifest = build_qlib_store_from_frame(
    qlib_store_frame,
    provider_uri=PROVIDER_URI,
    market='all',
    region='us',
    freq='day',
    field_map=field_map,
    as_of=SOURCE_END,
    source='data/processed/panels/soxl_soxx_vix_full.csv+data/raw/external/VIX9D,VIX3M,VVIX,SKEW',
)
shutil.copyfile(PROVIDER_URI / 'qlib_store_manifest.json', CARD / 'qlib_store_manifest.json')

provider_config = QlibProviderConfig(provider_uri=PROVIDER_URI, region='us', market='all', freq='day')
qlib_fields = ['$close', '$vix', '$vix9d', '$vix3m', '$vvix', '$skew', '$vix9d_vix3m', '$vvix_vix_z', '$skew_z']
qlib_audit_frame = load_qlib_feature_frame(
    provider_config,
    instruments=['SOXL'],
    fields=qlib_fields,
    start=feature_frame['datetime'].min().date().isoformat(),
    end=feature_frame['datetime'].max().date().isoformat(),
)

rows = []
def add_candidate(branch: str, candidate: str, exposure, params: dict) -> None:
    rows.append(
        evaluate_tail_risk_candidate(
            feature_frame,
            branch=branch,
            candidate=candidate,
            exposure=exposure,
            baseline_return_pct=baseline.return_pct,
            max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
            params=params,
            cost_per_turnover=COST_PER_TURNOVER,
        )
    )

def term_exposure(params: dict):
    risk = (
        (feature_frame['vix9d_vix3m'] < params['term'])
        & (feature_frame['vix9d_vix3m_z'] < params['termz'])
        & (feature_frame['vvix_vix_z'] < params['vvz'])
        & (feature_frame['skew_z'] < params['skz'])
        & (feature_frame['vix_z'] < params['vixz'])
    )
    trend = (feature_frame['trend_20'] > params['t20']) & (feature_frame['trend_60'] > params['t60'])
    return np.where(risk & trend, params['high'], np.where(trend, params['mid'], params['low']))

def vol_target_exposure(params: dict):
    raw = (params['target'] / (feature_frame['hv20'] + 1e-12)).clip(0, params['cap'])
    risk = (
        (feature_frame['vix9d_vix3m_z'] < params['termz'])
        & (feature_frame['vvix_vix_z'] < params['vvz'])
        & (feature_frame['skew_z'] < params['skz'])
        & (feature_frame['trend_20'] > params['t20'])
        & (feature_frame['trend_60'] > params['t60'])
    )
    return np.where(risk, raw, 0.0)

term_candidates = [
    ('term_high_return', {'t20': -0.2, 't60': -0.1, 'term': 0.86, 'termz': 1.0, 'vvz': 1.5, 'skz': 0.75, 'vixz': 0.0, 'high': 2.0, 'mid': 0.5, 'low': 0.2}),
    ('term_dd_gate', {'t20': -0.2, 't60': 0.05, 'term': 1.04, 'termz': 1.0, 'vvz': 0.75, 'skz': 0.0, 'vixz': 0.0, 'high': 0.5, 'mid': 0.175, 'low': 0.0}),
    ('term_medium', {'t20': -0.05, 't60': 0.05, 'term': 0.98, 'termz': 1.0, 'vvz': 0.0, 'skz': 0.0, 'vixz': 0.0, 'high': 0.75, 'mid': 0.2625, 'low': 0.0}),
]
for candidate, params in term_candidates:
    add_candidate('tail_risk_term_structure_ensemble', candidate, term_exposure(params), params)
add_candidate('tail_risk_term_structure_ensemble', 'term_cash_return_gate', np.zeros(len(feature_frame)), {'cash': True})

vol_candidates = [
    ('vol_high_return', {'target': 0.6, 'cap': 1.5, 'termz': 0.0, 'vvz': 1.5, 'skz': 2.0, 't20': -0.2, 't60': -0.2}),
    ('vol_dd_gate', {'target': 0.2, 'cap': 1.5, 'termz': 0.0, 'vvz': 1.5, 'skz': 2.0, 't20': -0.2, 't60': -0.2}),
    ('vol_low', {'target': 0.1, 'cap': 1.0, 'termz': 0.5, 'vvz': 1.0, 'skz': 1.5, 't20': -0.05, 't60': 0.0}),
]
for candidate, params in vol_candidates:
    add_candidate('vol_target_tail_risk_throttle', candidate, vol_target_exposure(params), params)
add_candidate('vol_target_tail_risk_throttle', 'vol_cash_return_gate', np.zeros(len(feature_frame)), {'cash': True})

candidate_metrics = pd.DataFrame(rows)
candidate_metrics.to_csv(CARD / 'agentic_alpha_tail_risk_external_candidate_metrics.csv', index=False)
branch_results = {
    branch: summarize_tail_risk_branch(
        candidate_metrics,
        branch=branch,
        baseline_return_pct=baseline.return_pct,
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    )
    for branch in sorted(candidate_metrics['branch'].unique())
}
passes_both = candidate_metrics.loc[candidate_metrics['beats_baseline_return'] & candidate_metrics['passes_drawdown_gate']]
best_candidate = None if passes_both.empty else str(passes_both.sort_values('return_pct', ascending=False).iloc[0]['candidate'])

search_summary = {
    'workflow': 'agentic_alpha_tail_risk_external_search',
    'stage_order': ['research', 'retrieve', 'generate', 'evaluate', 'validate', 'distill'],
    'baseline': {
        'name': baseline.name,
        'return_pct': float(baseline.return_pct),
        'max_drawdown_pct': float(baseline.max_drawdown_pct),
        'benchmark_return_pct': float(baseline.benchmark_return_pct),
        'recorder_id': baseline.recorder_id,
    },
    'goal_achieved': bool(best_candidate is not None),
    'best_candidate': best_candidate,
    'passes_both_count': int(len(passes_both)),
    'branch_results': branch_results,
    'external_source_counts': external_sources['source_type'].value_counts().sort_index().to_dict(),
    'external_data': {
        'source_fields': ['VIX9D', 'VIX3M', 'VVIX', 'SKEW'],
        'source_coverage_end': feature_frame.attrs['source_coverage_end'],
        'candidate_coverage_start': feature_frame['datetime'].min().date().isoformat(),
        'candidate_coverage_end': feature_frame['datetime'].max().date().isoformat(),
        'provider_manifest': 'data/qlib/soxl_tail_risk_external_v1/qlib_store_manifest.json',
        'feature_rows': int(len(feature_frame)),
        'qlib_audit_rows': int(len(qlib_audit_frame.reset_index())),
        'qlib_fields': qlib_fields,
    },
    'provider_manifest_hash': provider_manifest['hash'],
}
(CARD / 'agentic_alpha_tail_risk_external_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'failure_modes.md').write_text(
    '# Failure Modes\n\n'
    '- tail_risk_term_structure_ensemble: VIX9D/VIX3M, VVIX/VIX, and SKEW gates can cut risk, but high-return settings still accept deep SOXL drawdowns and drawdown-gated settings compound far below the production baseline.\n'
    '- vol_target_tail_risk_throttle: volatility targeting reduces MaxDD only by shrinking exposure so aggressively that return_gate fails.\n'
    '- coverage: VIX9D/VIX3M/VVIX/SKEW local files end on 2026-04-17, so this diagnostic cannot claim full 2026-06-09 production-window completion.\n'
    '- next_action: use a longer independent options-risk history, live flow features, or a validated model family instead of another threshold-only tail-risk throttle.\n',
    encoding='utf-8',
)

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(
        baseline=baseline.name,
        provider_uri=PROVIDER_URI.as_posix(),
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        source_coverage_end=feature_frame.attrs['source_coverage_end'],
        candidate_rows=len(candidate_metrics),
    )
    search_summary['recorder_id'] = recorder_id
    (CARD / 'agentic_alpha_tail_risk_external_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
    for path in [
        CARD / 'agentic_alpha_tail_risk_external_candidate_metrics.csv',
        CARD / 'agentic_alpha_tail_risk_external_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
        CARD / 'qlib_store_manifest.json',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_tail_risk_external')
    recorder_local_dir = Path(recorder.get_local_dir())

for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
search_summary['recorder_artifacts'] = recorder_artifacts
(CARD / 'agentic_alpha_tail_risk_external_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')

summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'promotion_decision': 'not_promoted' if not search_summary['goal_achieved'] else 'promote_candidate',
    'as_of': SOURCE_END,
    'data_range': [START, SOURCE_END],
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'recorder_artifacts': recorder_artifacts,
    'market': 'all',
    'region': 'us',
    'baseline': search_summary['baseline'],
    'goal_gate': {'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'require_return_above_baseline': True},
    'branch_results': branch_results,
    'search_report': search_summary,
    'external_source_counts': search_summary['external_source_counts'],
    'external_data': search_summary['external_data'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'model': 'https://qlib.readthedocs.io/en/latest/component/model.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start', 'build_qlib_store_from_frame'],
    'workflow_config': 'workflow.yaml',
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_tail_risk_external_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'qlib_store_manifest': 'qlib_store_manifest.json',
        'search_summary': 'agentic_alpha_tail_risk_external_search_summary.json',
    },
    'artifact_policy': {
        'policy': 'slim_card_recorder_first',
        'workflow_config': 'workflow.yaml',
        'machine_entrypoint': 'summary.json',
        'primary_artifact': f'{EXPERIMENT}.ipynb',
        'kept_sidecars': [
            'agentic_alpha_tail_risk_external_candidate_metrics.csv',
            'agentic_alpha_tail_risk_external_search_summary.json',
            'external_sources.csv',
            'failure_modes.md',
            'qlib_store_manifest.json',
        ],
        'recorder_only_artifacts': {
            'predictions': 'artifacts/pred.pkl',
            'labels': 'artifacts/label.pkl',
            'signals': 'artifacts/signals.pkl',
            'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
            'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
            'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
            'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        },
        'reproducible_from_recorder': True,
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'conclusion.md').write_text(
    '# 结论\n\n'
    f'- what_ran: 执行 {EXPERIMENT}，把 Github/论文/Arxiv/Twitter 的 R-G-E-V-D 外部知识映射到 VIX9D、VIX3M、VVIX、SKEW，并用 `build_qlib_store_from_frame` 构建 qlib provider 后通过 `D.features` 审计。\n'
    f'- key_metrics: production baseline return={baseline.return_pct:.2f}%, maxDD={baseline.max_drawdown_pct:.2f}%; 本卡 passes_both_count={len(passes_both)}。\n'
    f'- regime_window_difference: 外部尾部风险数据本地覆盖到 {feature_frame.attrs["source_coverage_end"]}，候选回测有效区间到 {feature_frame["datetime"].max().date().isoformat()}，不能冒充完整 2026-06-09 生产窗口。\n'
    '- promotion_decision: not_promoted，未找到同时满足收益超过现有生产策略且 MaxDD 小于 20% 的候选。\n'
    '- risks: term-structure 和 vol-target 阈值仍是日频门控，能降低部分尾部风险但会牺牲 SOXL 主要复利窗口；外部数据覆盖也短于生产基准。\n'
    '- next_action: 转向更长历史的期权尾部风险、实时资金流/订单流或新模型家族，不再重复单纯阈值门控。\n',
    encoding='utf-8',
)
print(json.dumps({'recorder_id': recorder_id, 'passes_both_count': int(len(passes_both)), 'branch_results': branch_results}, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[81652:MainThread](2026-06-23 17:15:25,836) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[81652:MainThread](2026-06-23 17:15:25,838) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[81652:MainThread](2026-06-23 17:15:25,839) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_tail_risk_external_v1')}


[81652:MainThread](2026-06-23 17:15:25,846) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[81652:MainThread](2026-06-23 17:15:25,943) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_tail_risk_external_search_v1.


[81652:MainThread](2026-06-23 17:15:25,944) INFO - qlib.workflow - [exp.py:258] - Experiment 219337719860073377 starts running ...


[81652:MainThread](2026-06-23 17:15:25,999) INFO - qlib.workflow - [recorder.py:345] - Recorder b83f8a70251a40d2b082ced2a2be0990 starts running under Experiment 219337719860073377 ...


[81652:MainThread](2026-06-23 17:15:26,062) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{
  "branch_results": {
    "tail_risk_term_structure_ensemble": {
      "best_drawdown_gate_candidate": "term_dd_gate",
      "best_drawdown_gate_max_drawdown_pct": -19.684469399146654,
      "best_drawdown_gate_return_pct": 142.71123390315012,
      "best_return_candidate": "term_high_return",
      "best_return_max_drawdown_pct": -80.64881104769321,
      "best_return_pct": 4652.052195135078,
      "coverage_end": "2026-04-16",
      "coverage_start": "2019-03-29",
      "goal_baseline_return_pct": 6570.0232821043,
      "grid_rows": 4,
      "max_drawdown_floor": -20.0,
      "passes_both_count": 0
    },
    "vol_target_tail_risk_throttle": {
      "best_drawdown_gate_candidate": "vol_dd_gate",
      "best_drawdown_gate_max_drawdown_pct": -19.157853796708725,
      "best_drawdown_gate_return_pct": 78.2904227296005,
      "best_return_candidate": "vol_high_return",
      "best_return_max_drawdown_pct": -49.833592645168736,
      "best_return_pct": 322.6716461163608,
      "coverage